# Core Numerical Methods Engine

## How Root Finding Works in Scientific Computing
In mathematics, a "root" of a function $f(x)$ is the value of $x$ where the 
function evaluates to zero:
$$
f(x) = 0
$$
In many real-world scientific problems, you cannot easily rearrange an equation 
with pen and paper to solve for $x$ algebraically because the equation is highly 
non-linear or relies on complex empirical lookup matrices.

A root-finding algorithm solves this by taking an initial guess for $x$ and 
iteratively evaluating the function, adjusting the guess step-by-step until 
$f(x)$ lands close enough to zero within an acceptable precision tolerance 
(e.g., $10^{-6}$).


## Applying Root-Finding to the Water Balance Equation

The core discrete water balance equation is:
$$S_{t+1} = S_t + R_t + I_t - ET_t - D_t$$

Let's assume we know our current soil state ($S_t$), today's rainfall ($R_t$), 
and today's weather inputs to calculate $ET_t$. We have a specific Target Soil 
Moisture ($S_{\text{target}}$) that we want to achieve tomorrow. We need to find 
the exact irrigation amount ($I_t$) that satisfies this goal.

The question now becomes:
> "How much irrigation water I must I apply today so that soil moisture S 
> reaches the target level $S_{\text{target}}$?"


To use a root-finding algorithm, we must rewrite this physical system as a 
function where our unknown variable is $x = I_t$, and our objective is to drive 
the difference between the simulated outcome and our target to zero:
$$f(I_t) = \text{Simulated } S_{t+1}(I_t) - S_{\text{target}} = 0$$

In [ ]:
# 1. Define the physical system model as our target function
def water_balance_objective(I_t, S_t, R_t, T, W, Solar, H, field_capacity, S_target):
    """
    Evaluates how far the resulting soil moisture is from our target 
    given a specific irrigation choice (I_t).
    Objective: Find I_t where this function returns 0.
    """
    # Calculate daily evapotranspiration
    et = max(0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)
    
    # Core water balance transformation
    S_next = S_t + R_t + I_t - et
    
    # Apply physical drainage boundary ceiling
    if S_next > field_capacity:
        S_next = field_capacity
    if S_next < 0:
        S_next = 0
        
    # Return the residual error gap we want to eliminate
    return S_next - S_target



# 2. Implement the Secant Root-Finding Algorithm from scratch
def solve_irrigation_secant(S_t, R_t, T, W, Solar, H, field_capacity, S_target, max_iter=50, tol=1e-5):
    """
    Uses the Secant Method to find the precise root (I_t) for our objective function.
    """
    # Two initial guesses for irrigation (e.g., 0 mm and 10 mm)
    x0 = 0.0
    x1 = 30.0
    
    for iteration in range(max_iter):
        # Evaluate function at both guess points
        f_x0 = water_balance_objective(x0, S_t, R_t, T, W, Solar, H, field_capacity, S_target)
        f_x1 = water_balance_objective(x1, S_t, R_t, T, W, Solar, H, field_capacity, S_target)
        
        # Prevent division by zero if the function flattens out
        if abs(f_x1 - f_x0) < 1e-12:
            break
            
        # Secant update formula
        x_next = x1 - f_x1 * (x1 - x0) / (f_x1 - f_x0)

        # This prevents the algorithm from falling into the flat zero-moisture zone
        if x_next < 0:
            x_next = 0.0
        
        # Check if the solution has converged close enough to zero
        if abs(water_balance_objective(x_next, S_t, R_t, T, W, Solar, H, field_capacity, S_target)) < tol:
            # Physical constraint: You cannot apply negative irrigation
            return max(0.0, x_next)
        
        # Update guess bounds for the next step
        x0 = x1
        x1 = x_next
        
    return max(0.0, x1)


Running a multiday simulation loop

Import the dataset and do data normalization and zone filtering

In [ ]:
import pandas as pd

weather = pd.read_csv('../data/raw/weather_daily.csv', 
                            na_values=['NA', ''])
soil = pd.read_csv('../data/raw/soil_sensor_data.csv', 
                              na_values=['NA', ''])


weather = weather.dropna()
soil = soil.dropna()

weather["date"] = pd.to_datetime(weather["date"]).dt.date
soil["date"] = pd.to_datetime(soil["timestamp"]).dt.date

soil_zone_a = soil[soil["zone_id"] == "Zone_A"].copy()
 
# Drop the original timestamp & zone_id columns 
soil_zone_a = soil_zone_a.drop(columns=["timestamp", "zone_id"])
 
# Inner merge on date
merged = pd.merge(weather, soil_zone_a, on="date", how="inner")
 
# Sort & reset index
merged = merged.sort_values("date").reset_index(drop=True)

In [ ]:
# 1. Setup global structural constants
field_capacity = 50.0    # Ceiling of soil water storage (mm)
S_target = 38.5         # Our optimal structural moisture goal (mm)

# Initialize the starting soil moisture on Day 1 from your actual sensor data
# (Assuming your sensor column is named 'soil_moisture_pct' and is in percentage)
current_S = merged.loc[0, 'soil_moisture_pct'] * field_capacity / 100.0  # Convert percentage to mm

print(f"Initial soil moisture (S) on Day 1: {current_S:.2f} mm")

# Pre-allocate lists to hold our simulated results
calculated_irrigation = []
simulated_S_history = []

# 2. Step through the dataset day by day
for idx, row in merged.iterrows():
    
    # Extract today's environmental inputs from the current row
    R_today = row['rainfall_mm']
    T_today = row['temperature_c']
    W_today = row['wind_speed_mps']
    Solar_today = row['solar_index']
    H_today = row['humidity_pct']
    
    # 3. Call the root-finder to compute the exact irrigation needed for TODAY
    # It takes today's starting moisture (current_S) and finds the matching water volume
    opt_I = solve_irrigation_secant(
        current_S, R_today, T_today, W_today, Solar_today, H_today, field_capacity, S_target
    )
    
    # Record the computed irrigation choice
    calculated_irrigation.append(opt_I)
    
    # 4. Update the physics engine to calculate the true closing state of the day
    # We pass the optimal irrigation value back into the balance equation
    et_today = max(0, 0.12*T_today + 0.35*W_today + 2.4*Solar_today - 0.025*H_today)
    
    # Soil moisture calculation adding today's rain and calculated optimal water
    next_S = current_S + R_today + opt_I - et_today
    
    # Apply physical clipping boundaries
    if next_S > field_capacity:
        next_S = field_capacity
    if next_S < 0:
        next_S = 0
        
    # Record the ending soil storage state
    simulated_S_history.append(next_S)
    
    # CRITICAL STEP: Tomorrow's starting moisture is today's closing moisture
    current_S = next_S

# 5. Append the calculated arrays directly back into your main DataFrame
merged['Precision_Irrigation_Need'] = calculated_irrigation
merged['Simulated_Soil_State'] = simulated_S_history

# View the aggregated results
# Inspect the first few rows of the updated precision strategy
print(merged[['date', 'soil_moisture_pct', 'Precision_Irrigation_Need', 'Simulated_Soil_State']])

We loop sequentially instead of vectorizing the root-finder due to 
**temporal dependency (system memory)**. In Level 2, calculating 
evapotranspiration ($ET$) only required looking at that day's independent 
weather parameters. In Level 3, however, today's starting moisture state 
($S_t$) is explicitly defined by yesterday's final calculated moisture state 
($S_{t-1}$). Because the rows are linked in a chain, the computer must solve 
the roots sequentially day by day down the line.

### Root Finding Methods

#### 1. The Bisection Method

The Bisection Method is a **bracketing** or **closed** method used to find a 
real root of a continuous function $f(x) = 0$. It relies on the 
**Intermediate Value Theorem**, which states that if a continuous function 
changes signs over an interval $[a, b]$ (i.e., $f(a) \times f(b) < 0$), there 
must be at least one root $c$ within that interval.

1. **Initialization:** Choose a lower bound $a$ and an upper bound $b$ such 
that $f(a)$ and $f(b)$ have opposite signs.
2. **Midpoint Calculation:** Compute the midpoint of the interval:

$$c = \frac{a + b}{2}$$


3. **Evaluation:** Check if $c$ is the root (i.e., $f(c) = 0$ or within a tiny 
tolerance $\epsilon$).
4. **Interval Reduction:** * If $f(a) \times f(c) < 0$, the root lies in the 
lower half. Set $b = c$.
* If $f(b) \times f(c) < 0$, the root lies in the upper half. Set $a = c$.


5. **Iteration:** Repeat steps 2–4 until the interval width $|b - a|$ or 
$|f(c)|$ is smaller than your predefined tolerance.

* **Pros:** Guaranteed to converge if the function is continuous and bounds 
bracket a root.
* **Cons:** Converges slowly (linear convergence rate).


#### 2. The Newton-Raphson Method

The Newton-Raphson method is an **open** method, meaning it requires only a 
single initial guess $x_0$ rather than a bounded bracket. It utilizes calculus 
to approximate the curve linearly at each step.

1. **Initialization:** Start with an initial guess $x_0$ near the true root.
2. **Linear Approximation:** Draw a tangent line to the function graph at 
$(x_i, f(x_i))$. The slope of this line is the derivative $f'(x_i)$.
3. **Finding the Intercept:** Find where this tangent line crosses the x-axis. 
This x-intercept becomes the next, improved approximation $x_{i+1}$. 
Mathematically derived from the equation of a line:

$$x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)}$$


4. **Iteration:** Repeat the process substituting $x_{i+1}$ for $x_i$ until 
$|x_{i+1} - x_i|$ or $|f(x_{i+1})|$ drops below the tolerance limit.

* **Pros:** Extremely fast when it works (quadratic convergence rate—the number 
of correct decimal places roughly doubles with each iteration).
* **Cons:** Requires knowing the analytical derivative $f'(x)$. It can fail or 
diverge completely if $f'(x_i) \approx 0$ (a flat slope sends the next guess to 
infinity) or if the initial guess is too far from the root.

#### 3. The Secant Method

The Secant Method can be thought of as a variant of the Newton-Raphson method 
designed for scenarios where finding the analytical derivative $f'(x)$ is too 
difficult or computationally expensive. Instead of calculating a tangent line 
based on a derivative at one point, it projects a **secant line** through two 
historical points.

1. **Initialization:** Choose two initial guesses, $x_0$ and $x_1$ (they do not 
need to bracket the root).
2. **Secant Approximation:** Approximate the derivative using the slope between 
the two points:

$$f'(x_1) \approx \frac{f(x_1) - f(x_0)}{x_1 - x_0}$$


3. **Update Formula:** Substitute this finite-difference approximation into the Newton-Raphson update equation to find the next point $x_{i+1}$:

$$x_{i+1} = x_i - f(x_i) \frac{x_i - x_{i-1}}{f(x_i) - f(x_{i-1})}$$


4. **Iteration:** Discard the oldest point ($x_{i-1}$), retain the newest two, 
and loop until convergence.

* **Pros:** No derivative function $f'(x)$ required; faster than Bisection 
(superlinear convergence rate, roughly $\approx 1.62$).
* **Cons:** Still prone to divergence if the function behaves erratically or if 
$f(x_i) \approx f(x_{i-1})$.

### Numerical Differentiation

In numerical computing, finite difference methods approximate derivatives 
(rates of change) using discrete data points separated by a specific step size 
$h$ (the time step).

#### 1. Forward Difference

The forward difference method estimates the slope at a current point $x_i$ by 
looking ahead to the next consecutive data point $x_{i+1}$.


$$\text{Formula: } \quad f'(x_i) \approx \frac{f(x_{i+1}) - f(x_i)}{h}$$

* **Truncation Error:** First-order accurate, $O(h)$.
* **Limitation:** It cannot be computed for the final point in a dataset since 
there is no "forward" point left.

#### 2. Backward Difference

The backward difference method estimates the slope at the current point $x_i$ 
by looking behind at the previous data point $x_{i-1}$.


$$\text{Formula: } \quad f'(x_i) \approx \frac{f(x_i) - f(x_{i-1})}{h}$$

* **Truncation Error:** First-order accurate, $O(h)$.
* **Limitation:** It cannot be computed for the first point in a dataset since 
there is no "backward" historical point.

#### 3. Central Difference

The central difference method averages the spacing by taking a point ahead 
$x_{i+1}$ and a point behind $x_{i-1}$, completely bypassing the actual value 
at $x_i$.


$$\text{Formula: } \quad f'(x_i) \approx \frac{f(x_{i+1}) - f(x_{i-1})}{2h}$$

* **Truncation Error:** Second-order accurate, $O(h^2)$. This makes it 
significantly more precise than forward or backward differences when using a 
small step size.
* **Limitation:** It cannot be computed for either the first or last boundaries 
of a dataset.



#### Estimating Soil Moisture Change

Instead of writing a manual for loop we use pandas vectorization with the 
`.shift()` function which is the standard, most efficient way to compute finite 
differences directly on a DataFrame without loops.

In [ ]:
import pandas as pd

# 1. Read the CSV directly and parse the timestamp column as dates
soil = pd.read_csv('../data/raw/soil_sensor_data.csv', 
                              na_values=['NA', ''], parse_dates=["timestamp"])

# 2. Extract the rows you need (Filter for Zone_A and sort chronologically)
target_zone = "Zone_A"
df_zone = (
    soil[soil["zone_id"] == target_zone].sort_values("timestamp").reset_index(drop=True)
)

# Define step size (h) in hours (daily readings = 24 hours)
h = 24.0

# 3. Directly calculate differences using pandas vectorization (.shift)
# .shift(-1) gets the next row's value (t + 1)
# .shift(1) gets the previous row's value (t - 1)

df_zone["Forward_Diff"] = (
    df_zone["soil_moisture_pct"].shift(-1) - df_zone["soil_moisture_pct"]
) / h
df_zone["Backward_Diff"] = (
    df_zone["soil_moisture_pct"] - df_zone["soil_moisture_pct"].shift(1)
) / h
df_zone["Central_Diff"] = (
    df_zone["soil_moisture_pct"].shift(-1) - df_zone["soil_moisture_pct"].shift(1)
) / (2 * h)

# 4. Display the resulting rows with the new calculation columns
columns_to_show = [
    "timestamp",
    "soil_moisture_pct",
    "Forward_Diff",
    "Backward_Diff",
    "Central_Diff",
]
print(df_zone[columns_to_show])

#### Summary and Analysis

* **Boundary Conditions:** Notice that at the start of the dataset (`Index 0` on `2026-03-01`), the backward and central differences are unavailable (`NaN`) 
because there is no historical data point preceding it. Conversely, at the very 
end of the observed sequence (`Index 29` on `2026-03-30`), the forward and 
central differences are marked as `NaN` because there is no subsequent data 
point available to calculate a forward trajectory.
* **Central Difference Accuracy:** Look closely at `Index 2` (`2026-03-03`). The 
forward difference predicts a rate of change of `-0.079167 %/hr`, while the 
backward difference calculated `-0.100000 %/hr`. The central difference takes 
both surrounding trends into account, outputting a second-order accurate 
localized approximation of `-0.089583 %/hr`. The negative signs across all three 
metrics at this timestamp correctly indicate that the soil is continuously 
drying out during this localized observation window.

### Numerical Integration

In numerical computing, the **Trapezoidal Rule** and **Simpson's Rule** are 
popular techniques for numerical integration (often called 
*numerical quadrature*). Instead of finding an analytical antiderivative, these 
methods calculate the definite integral, the area under a curve, by breaking 
the total domain down into small, manageable segments using discrete data 
points.

#### 1. The Trapezoidal Rule

The Trapezoidal Rule approximates the area under a curve by connecting 
consecutive data points with a **straight line** (a first-degree linear 
polynomial). This transforms each slice of area into a geometric trapezoid.

For a single segment over an interval $[x_i, x_{i+1}]$ with a step width of 
$h$, the area of the trapezoid is:


$$\text{Area} = h \times \frac{f(x_i) + f(x_{i+1})}{2}$$

When summing all segments across a uniform grid from $x_0$ to $x_n$, the edge 
boundaries are counted once, while all internal node values are shared by 
adjacent trapezoids and counted twice:


$$\int_{a}^{b} f(x) \, dx \approx \frac{h}{2} \left[ f(x_0) + 2\sum_{i=1}^{n-1} 
f(x_i) + f(x_n) \right]$$

* **Error Profile:** It has a truncation error of $O(h^2)$.
* **Data Constraints:** Extremely flexible. It can handle any number of data 
points, and the step size $h$ between points does not even need to be uniform.


#### 2. Simpson's Rule (Simpson's 1/3 Rule)

Instead of connecting points with rigid straight lines, Simpson's Rule matches 
a **parabolic arc** (a second-degree quadratic polynomial) across the data 
points. Because a parabola requires three points to be uniquely defined, 
Simpson's Rule evaluates pairs of adjacent intervals simultaneously.

For any two adjacent segments spanning $[x_{i-1}, x_{i+1}]$ with a uniform step 
size $h$, the area under the quadratic curve is:


$$\text{Area} = \frac{h}{3} \left[ f(x_{i-1}) + 4f(x_i) + f(x_{i+1}) \right]$$

When compiled across an entire uniform dataset, the composite formula alternates 
weights of $4$ and $2$ for the inner nodes:


$$\int_{a}^{b} f(x) \, dx \approx \frac{h}{3} \left[ f(x_0) 
+ 4\sum_{i=1,3,5\dots}^{n-1} f(x_i) + 2\sum_{j=2,4,6\dots}^{n-2} f(x_j) 
+ f(x_n) \right]$$

* **Error Profile:** It features a higher-order truncation error of $O(h^4)$, 
making it significantly more accurate than the Trapezoidal Rule for smooth 
functions.
* **Data Constraints:** It **strictly requires a uniform step size ($h$)** and 
an **odd number of data points** (which translates to an 
*even number of intervals* $n$).

### Estimating Cumulative Water Deficit in Python

In agricultural engineering, hydrology, and smart irrigation systems, the 
**water deficit** is the difference between water lost through 
evapotranspiration and water gained through rainfall. Integrating this deficit 
over time gives the **cumulative water deficit**, which tells engineers exactly 
how much irrigation water must be added back to preserve crops.

Because the dataset spans 30 days (2026-03-01 to 2026-03-30), we have 30 data 
points, which means 29 intervals.

* The Trapezoidal Rule works natively with any number of intervals.

* Simpson's 1/3 Rule strictly requires an even number of intervals (odd number 
of points). To handle this real-world constraint, the standard practice in 
scientific computing is to integrate the first 28 intervals (the first 29 
points) using Simpson's Rule, and then add the final remaining interval using 
the Trapezoidal Rule.

In [ ]:
import numpy as np
import pandas as pd

def trapezoidal_rule(y_values, step_size):
    """Integrates over all intervals using the Trapezoidal Rule."""
    n = len(y_values) - 1
    total_area = y_values[0] + y_values[-1]

    for i in range(1, n):
        total_area += 2 * y_values[i]

    return (step_size / 2.0) * total_area


def composite_simpsons_rule(y_values, step_size):
    """Integrates using Simpson's 1/3 Rule.

    If intervals are odd, handles the last interval with a Trapezoidal step.
    """
    n = len(y_values) - 1

    # Check if intervals are odd
    if n % 2 != 0:
        # Separate the main body (even number of intervals) from the last point
        simps_y = y_values[:-1]
        trap_y = y_values[-2:]  # Last interval (points n-1 to n)

        # Integrate both sections
        simps_area = simpsons_core(simps_y, step_size)
        trap_area = (step_size / 2.0) * (trap_y[0] + trap_y[1])

        return simps_area + trap_area
    else:
        return simpsons_core(y_values, step_size)


def simpsons_core(y_values, step_size):
    """Core Simpson's 1/3 implementation for even intervals."""
    n = len(y_values) - 1
    total_area = y_values[0] + y_values[-1]

    for i in range(1, n):
        if i % 2 == 1:
            total_area += 4 * y_values[i]  # Odd indices weight 4
        else:
            total_area += 2 * y_values[i]  # Even indices weight 2

    return (step_size / 3.0) * total_area



weather = pd.read_csv('../data/raw/weather_daily.csv', 
                            na_values=['NA', ''], parse_dates=["date"])

# Fill missing rainfall values with 0.0 (assuming no rain recorded) or drop/interpolate
weather["rainfall_mm"] = pd.to_numeric(weather["rainfall_mm"], errors="coerce").fillna(0.0)

# Extract values into an array
rainfall = weather["rainfall_mm"].tolist()
h = 1.0  # Constant step size of 1 day

# ==========================================
# 3. COMPUTE AND PRINT RESULTS
# ==========================================
total_rain_trap = trapezoidal_rule(rainfall, h)
total_rain_simp = composite_simpsons_rule(rainfall, h)

print('HYDROLOGICAL INTEGRATION REPORT:')
print(f"Total Date Records       : {len(weather)} days")
print(f"Total Intervals (n)      : {len(weather) - 1} intervals (ODD)")
print(f"Step Size (h)            : {h} day")
print(f"Trapezoidal Total Rain   : {total_rain_trap:.3f} mm")
print(f"Simpson's Mix Total Rain : {total_rain_simp:.3f} mm")
print(f"Absolute Difference      : {abs(total_rain_trap - total_rain_simp):.4f} mm")

#### Hydrological Integration Summary

* **Dataset Constraints and Structure:** The dataset spans a duration of 
**30 days** with a uniform step size ($h$) of **1.0 day**. This translates to 
exactly **29 discrete intervals**. Because 29 is an **odd number**, standard 
Simpson's 1/3 rule cannot be applied uniformly across the entire dataset without architectural modifications.
* **Algorithmic Discrepancy (Trapezoidal vs. Simpson's Mix):** The pure 
Trapezoidal Rule estimates the cumulative rainfall to be **245.350 mm**. In 
contrast, the Simpson's Mix approach (which utilizes second-order parabolic 
approximations across the first 28 intervals and a linear trapezoidal patch for 
the final odd interval) yields a higher estimate of **265.917 mm**.
* **Error and Variance Evaluation:** An absolute difference of **20.5667 mm** 
exists between the two numerical integration methods. Because the dataset 
contains sharp, non-linear rainfall spikes (such as the massive 85.00 mm event 
observed in the project data), the Trapezoidal Rule tends to underestimate or 
blunt the peak areas under the curve. The Simpson's Mix calculation offers a 
significantly more robust, higher-order fit for these volatile fluctuations, 
explaining the ~20.5 mm variance.

### Three Zone Water Allocation Using LU Decomposition

In [ ]:
import numpy as np
import pandas as pd

soil = pd.read_csv('../data/raw/soil_sensor_data.csv', 
                              na_values=['NA', ''])

# Clean numerical anomalies: convert pump flow to numeric, dropping strings like 'CHECK'
soil["pump_flow_lpm"] = pd.to_numeric(soil["pump_flow_lpm"], errors="coerce")
soil["pump_power_watts"] = pd.to_numeric(soil["pump_power_watts"], errors="coerce")

# Extract aggregate constants from zones to build vector b (e.g., mean flow rates)
mean_flow_A = soil[soil["zone_id"] == "Zone_A"]["pump_flow_lpm"].mean(skipna=True)
mean_flow_B = soil[soil["zone_id"] == "Zone_B"]["pump_flow_lpm"].mean(skipna=True)
mean_flow_C = soil[soil["zone_id"] == "Zone_C"]["pump_flow_lpm"].mean(skipna=True)

# matrix A (Zone interconnected flow coefficients)
A = np.array([[2.0, -1.0, 0.0], [-1.0, 2.0, -1.0], [0.0, -1.0, 2.0]])

# vector b using real sensor aggregates
b = np.array([mean_flow_A, mean_flow_B, mean_flow_C])


x_direct = np.linalg.solve(A, b)


print(f"[FINAL WATER ALLOCATION RESULTS: {x_direct}")


# Verify the solution matches perfectly by back-multiplying: A * x = b
verification = np.allclose(np.dot(A, x_direct), b)
print(f"Solution Mathematically Valid?   : {verification}")